# Session 2 Solution: Environment Setup and SDK Initialization

In [2]:
from pathlib import Path
import sys

candidates = [
    Path.cwd() / "solutions" / "src",
    Path.cwd().parent / "src",
    Path.cwd() / "src",
    Path.cwd().parent.parent / "solutions" / "src"
]
for path in candidates:
    if path.exists() and str(path) not in sys.path:
        sys.path.append(str(path))

from hackathon_solutions.config import load_config, validate_config
from hackathon_solutions.evaluation_runner import run_basic_connectivity_check

cfg = load_config()
validate_config(cfg)
print("Config loaded for project endpoint:", cfg.project_endpoint)

Config loaded for project endpoint: https://edwinfoundry0226.services.ai.azure.com/api/projects/proj-default


In [3]:
result = run_basic_connectivity_check()
result

{'project_endpoint': 'https://edwinfoundry0226.services.ai.azure.com/api/projects/proj-default',
 'agent_count': 0,
 'agent_ids': []}

In [4]:
import base64
import json
from datetime import datetime, timezone

from azure.identity import DefaultAzureCredential


def _decode_jwt_payload(access_token: str) -> dict:
    payload_b64 = access_token.split(".")[1]
    padding = "=" * (-len(payload_b64) % 4)
    payload = base64.urlsafe_b64decode(payload_b64 + padding).decode("utf-8")
    return json.loads(payload)


credential = DefaultAzureCredential()
# Preferred scope for Azure AI Foundry auth checks.
token = credential.get_token("https://ai.azure.com/.default")
claims = _decode_jwt_payload(token.token)

sanitized = {
    "aud": claims.get("aud"),
    "tenant_id": claims.get("tid"),
    "client_app_id": claims.get("appid") or claims.get("azp"),
    "expires_utc": datetime.fromtimestamp(token.expires_on, tz=timezone.utc).isoformat(),
}

print("Entra token acquired successfully (sanitized).")
print(json.dumps(sanitized, indent=2))

Entra token acquired successfully (sanitized).
{
  "aud": "https://ai.azure.com",
  "tenant_id": "c3b197bb-2c04-4ad6-8f93-024e4c5106dd",
  "client_app_id": "03b19d4e-dfe3-440b-88f4-db6b3feaf0a3",
  "expires_utc": "2026-03-15T23:30:51+00:00"
}
